# Phase 6 — PPO corrigé (RLHF étape 2, version fixée)

Relance le PPO avec les correctifs : symétrie policy/ref, dropout désactivé au scoring, KL adaptatif + whitening des récompenses, 1 époque PPO.

## ⚠️ Datasets à attacher (DEUX)
- `adl-dpo-adapter` (notebook 01)
- `adl-reward-model` (notebook 02)

**Setup** : GPU T4 x1, Internet On. **Durée** : ~4-6 h.

> Prérequis : avoir poussé `training/train_ppo_fixed.py` sur le repo GitHub.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
# Vérifie que les scripts corrigés sont bien dans le repo cloné.
# Si ça échoue : commit + push des 4 fichiers sur ton repo GitHub d'abord.
import os
for p in ["eval/kl_diagnostics.py","training/train_ppo_fixed.py",
          "eval/evaluate_ethics_fixed.py","eval/stats_analysis.py"]:
    assert os.path.isfile(p), f"MANQUANT: {p} — pousse les scripts corrigés sur GitHub."
print("Scripts corrigés présents.")

In [ ]:
DPO_ZIP = "/kaggle/input/adl-dpo-adapter/dpo_model.zip"
RM_ZIP  = "/kaggle/input/adl-reward-model/reward_model.zip"
import os, zipfile
os.makedirs("results/dpo_model", exist_ok=True)
os.makedirs("results/reward_model", exist_ok=True)
with zipfile.ZipFile(DPO_ZIP) as z: z.extractall("results/dpo_model")
with zipfile.ZipFile(RM_ZIP)  as z: z.extractall("results/reward_model")
print("DPO:", os.listdir("results/dpo_model"))
print("RM:",  os.listdir("results/reward_model"))

In [ ]:
!python data/prepare_preferences.py --n_pku 15000 --n_ultra 5000 \
    --out_path data/preferences.jsonl

In [ ]:
!python training/train_ppo_fixed.py \
    --dpo_adapter results/dpo_model \
    --reward_model_path results/reward_model \
    --data_path data/preferences.jsonl \
    --output_dir results/rlhf_model_fixed

## Export

Garde le `checkpoint-*/trainer_state.json` dans le zip : il contient la nouvelle courbe de KL pour la figure 1 corrigée.

In [ ]:
import shutil, os
src = "/kaggle/working/adl/results/rlhf_model_fixed"
out = "/kaggle/working/rlhf_model_fixed.zip"
shutil.make_archive(out.replace(".zip",""), "zip", src)
print(f"Zipped -> {out}  ({os.path.getsize(out)//1024//1024} MB)")
print("Crée un Kaggle Dataset 'adl-rlhf-model-fixed' à attacher au notebook 07")